[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-2/chatbot-external-memory.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239440-lesson-6-chatbot-w-summarizing-messages-and-external-memory)

# Chatbot with message summarization & external DB memory

## 復習

我們已經介紹過如何自訂 graph state 的 schema 與 reducer。
 
我們也示範了多種在 graph state 中修剪（trim）或過濾（filter）message 的技巧。

我們把這些概念用在一個具備 memory 的 Chatbot 上，讓它能持續產生對話的即時摘要。

## 目標

但是，如果我們希望 Chatbot 擁有可以無限期持續保存的 memory 呢？

接下來，我們會介紹一些更進階、支援外部資料庫的 checkpointer。

在這裡，我們會示範如何使用 [Sqlite as a checkpointer](https://docs.langchain.com/oss/python/langgraph/persistence#checkpointer-libraries)，不過其他 checkpointer（例如 Postgres）同樣可以使用！

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph-checkpoint-sqlite langchain_core langgraph langchain_openai

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

## Sqlite

在這裡，一個很好的起點是 [SqliteSaver checkpointer](https://docs.langchain.com/oss/python/langgraph/persistence#checkpointer-libraries)。

Sqlite 是一個 [小巧、快速、非常受歡迎的](https://x.com/karpathy/status/1819490455664685297) SQL 資料庫。
 
如果我們傳入 `":memory:"`，它會建立一個 in-memory 的 Sqlite 資料庫。

In [ ]:
import sqlite3
# in-memory 資料庫
conn = sqlite3.connect(":memory:", check_same_thread = False)

但是，如果我們傳入一個 db path，它就會幫我們建立一個資料庫！

In [ ]:
# 若檔案不存在則下載，並連線到本機 db
!mkdir -p state_db && [ ! -f state_db/example.db ] && wget -P state_db https://github.com/langchain-ai/langchain-academy/raw/main/module-2/state_db/example.db

db_path = "state_db/example.db"
conn = sqlite3.connect(db_path, check_same_thread=False)

In [ ]:
# 這就是我們的 checkpointer 
from langgraph.checkpoint.sqlite import SqliteSaver
memory = SqliteSaver(conn)

讓我們重新定義我們的 chatbot。

In [ ]:
from typing_extensions import Literal
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, RemoveMessage

from langgraph.graph import END
from langgraph.graph import MessagesState


model = ChatOpenAI(model="gpt-4o",temperature=0)

class State(MessagesState):
    summary: str

# 定義呼叫 model 的邏輯
def call_model(state: State):
    
    # 取得既有的 summary（若存在）
    summary = state.get("summary", "")

    # 如果有 summary，就把它加進去
    if summary:
        
        # 把 summary 加進 system message
        system_message = f"Summary of conversation earlier: {summary}"

        # 把 summary 接在所有較新的 message 前面
        messages = [SystemMessage(content=system_message)] + state["messages"]
    
    else:
        messages = state["messages"]
    
    response = model.invoke(messages)
    return {"messages": response}

def summarize_conversation(state: State):
    
    # 首先，我們取得既有的 summary
    summary = state.get("summary", "")

    # 建立我們的摘要 prompt 
    if summary:
        
        # summary 已經存在
        summary_message = (
            f"This is summary of the conversation to date: {summary}\n\n"
            "Extend the summary by taking into account the new messages above:"
        )
        
    else:
        summary_message = "Create a summary of the conversation above:"

    # 把 prompt 加進我們的對話歷史
    messages = state["messages"] + [HumanMessage(content=summary_message)]
    response = model.invoke(messages)
    
    # 除了最近的 2 則 message 之外，其餘全部刪除
    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"summary": response.content, "messages": delete_messages}

# 判斷要結束對話，還是要對對話進行摘要
def should_continue(state: State)-> Literal ["summarize_conversation",END]:
    
    """回傳下一個要執行的 node。"""
    
    messages = state["messages"]
    
    # 如果 message 超過六則，我們就對對話進行摘要
    if len(messages) > 6:
        return "summarize_conversation"
    
    # 否則就直接結束
    return END

現在，我們只要用 sqlite checkpointer 重新編譯（re-compile）即可。

In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START

# 定義一個新的 graph
workflow = StateGraph(State)
workflow.add_node("conversation", call_model)
workflow.add_node(summarize_conversation)

# 把進入點（entrypoint）設為 conversation
workflow.add_edge(START, "conversation")
workflow.add_conditional_edges("conversation", should_continue)
workflow.add_edge("summarize_conversation", END)

# 編譯
graph = workflow.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

現在，我們可以多次呼叫（invoke）這個 graph。

In [ ]:
# 建立一個 thread
config = {"configurable": {"thread_id": "1"}}

# 開始對話
input_message = HumanMessage(content="hi! I'm Lance")
output = graph.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

input_message = HumanMessage(content="what's my name?")
output = graph.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

input_message = HumanMessage(content="i like the 49ers!")
output = graph.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

讓我們確認一下 state 確實有被存在本機。

In [ ]:
config = {"configurable": {"thread_id": "1"}}
graph_state = graph.get_state(config)
graph_state

### Persisting state

使用像 Sqlite 這樣的資料庫，意味著 state 會被持久化保存（persisted）！

舉例來說，我們可以重新啟動 notebook 的 kernel，並且看到我們依然能從磁碟上的 Sqlite DB 載入資料。


In [ ]:
# 建立一個 thread
config = {"configurable": {"thread_id": "1"}}
graph_state = graph.get_state(config)
graph_state

## Studio

**⚠️ 注意**

在拍攝這些影片之後，我們更新了 Studio，現在它可以在本機執行並透過你的瀏覽器存取。這是執行 Studio 的建議方式，取代影片中所示範的桌面版 App（Desktop App）。它現在被稱為 _LangSmith Studio_，而不再是 _LangGraph Studio_。詳細的設定說明可以在課程開頭的「Getting Setup」指南中找到。你可以在[這裡](https://docs.langchain.com/langsmith/studio)找到 Studio 的說明，並在[這裡](https://docs.langchain.com/langsmith/quick-start-studio#local-development-server)找到本機部署的具體細節。  
若要啟動本機開發伺服器，請在本模組的 `/studio` 目錄下，在你的終端機執行以下指令：

```
langgraph dev
```

你應該會看到以下輸出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打開你的瀏覽器，並前往上面顯示的 **Studio UI** 網址。
載入 Studio 中的 `chatbot`，它使用的是在 `module-2/studio/langgraph.json` 中設定的 `module-2/studio/chatbot.py`。